### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [3]:
# Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.\nBM25 and vector-based retrieval can be combined in LangChain to support hybrid retrieval strategies

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [5]:
# Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
### Step 3: Create MMR Retirever
retriever=vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

In [10]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")
llm=init_chat_model("openai:o4-mini-2025-04-16")
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000022929B32350>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022929B32710>, root_client=<openai.OpenAI object at 0x0000022929B31D10>, root_async_client=<openai.AsyncOpenAI object at 0x0000022929B32490>, model_name='o4-mini-2025-04-16', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [11]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [12]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain provides first-class support for both “agents” and “memory” as two orthogonal building blocks you can drop into your chains.  

1. Agents  
   • An “agent” in LangChain is really just an LLM wrapper that, at each step, decides which external tool to invoke (and in what order) based on the user’s input and the current context.  
   • Tools can be anything from a calculator API or a web-search interface to your own custom Python function. You register them with the agent, and the agent’s planning logic (powered by the LLM) figures out which tool calls to make.  
   • Behind the scenes you get an AgentExecutor which orchestrates LLM prompts, tool invocation, result handling, and final answer composition.  

2. Memory  
   • LangChain’s Memory modules let you persist conversation‐level context across turns so your chains and agents can “remember” what came before.  
   • ConversationBufferMemory stores the full chat history verbatim, making it available in each new pro

In [13]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='f9980324-bce9-4c68-aa13-ce72c6ed358f', metadata={'source': 'langchain_rag_dataset.txt'}, page_content="The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.\nThe 'refine' chain iteratively updates an answer by incorporating each new chunk of information.\nLangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory."),
  Document(id='7c06c241-7f25-45e7-be91-a189ce048ed6', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgen